In [3]:



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from SamplingRVs import runsamples


# Settings
numtimes       = 10      # iterations (rows)
numruns        = 100     # runs per iteration (averaged within iteration)
numsamples     = 69
orbsamplerange = [3, 14]
samplim        = 7

displayCDF  = False
displayElse = False
elseError   = False
saveElse    = False

SaveFig     = False
PrintTest   = False
savefigname = "CDF_Paper"

# =====================
# Scenario Labels
#   F = Full sample; L = limited (≥ samplim)
#   ∞, 1000, 30 = period caps; TL = tidally locked
# =====================
scenario_labels = {
    0: "F-∞",
    1: "F-1000",
    2: "F-30",
    3: "F-TL",
    4: "L-∞",
    5: "L-1000",
    6: "L-30",
    7: "L-TL",
}
scenario_order = [scenario_labels[k] for k in range(8)]


records = []  # (iteration, run, scenario, comp, stat, value)

for t in range(numtimes):
    print(f'Iteration {t+1}')
    for r in range(numruns):
        run = r + 1
        mKSval, aKSval, mpval, apval = runsamples(
            numsamples, orbsamplerange, samplim, run,
            displayCDF, displayElse, elseError, saveElse,
            SaveFig, PrintTest, savefigname
        )
        plt.close('all')  # in case runsamples made plots

        # 8 scenarios per list
        for k in range(8):
            scen = scenario_labels[k]
            # Observed (measured) vs Expected (actual)
            records.append((t+1, run, scen, "Observed", "KS", mKSval[k]))
            records.append((t+1, run, scen, "Observed", "p",  mpval[k]))
            records.append((t+1, run, scen, "Expected", "KS", aKSval[k]))
            records.append((t+1, run, scen, "Expected", "p",  apval[k]))

df = pd.DataFrame(records, columns=["iteration", "run", "scenario", "comp", "stat", "value"])


summary_iter = (
    df.groupby(["iteration", "scenario", "comp", "stat"])["value"]
      .agg(mean="mean", std=lambda x: x.std(ddof=1))
      .reset_index()
)
print("========== End of Line ==========")

Iteration 1


Run 1: 100%|██████████| 69/69 [00:00<00:00, 101.63it/s]


Total number of Star observations: 69
Single peak stars (SB1): 57
Double peak stars (SB2): 12



Run 100: 100%|██████████| 69/69 [00:00<00:00, 106.63it/s]


Iteration 2


...

Iteration 10


Run 1: 100%|██████████| 69/69 [00:00<00:00, 99.54it/s] 


Total number of Star observations: 69
Single peak stars (SB1): 57
Double peak stars (SB2): 12



Run 100: 100%|██████████| 69/69 [00:00<00:00, 102.91it/s]


========== End of Line ==========


In [4]:


def make_table_with_summary(summary_iter, df_raw, comp, stat, scenario_order, numtimes, decimals=3):
    """
    Build a table with mean +/- std per iteration and three summary rows:
    - All A: mean±std of iteration means
    - All B: pooled across all runs
    - All C: law of total variance (within + between)
    """
    sub = summary_iter[(summary_iter["comp"] == comp) & (summary_iter["stat"] == stat)]

    # Pivot numeric mean/std
    pivot_mean = sub.pivot_table(
        index="iteration", columns="scenario", values="mean", aggfunc="first"
    ).reindex(columns=scenario_order, index=range(1, numtimes+1))
    pivot_std = sub.pivot_table(
        index="iteration", columns="scenario", values="std", aggfunc="first"
    ).reindex(columns=scenario_order, index=range(1, numtimes+1))

    # Format each cell as mean±std (iteration-level)
    formatted = pd.DataFrame(index=pivot_mean.index, columns=scenario_order)
    for scen in scenario_order:
        for it in pivot_mean.index:
            if pd.notnull(pivot_mean.loc[it, scen]):
                mu = pivot_mean.loc[it, scen]
                sd = pivot_std.loc[it, scen]
                formatted.loc[it, scen] = f"{mu:.{decimals}f} ± {sd:.{decimals}f}"
            else:
                formatted.loc[it, scen] = "—"


    summary_A, summary_B, summary_C = {}, {}, {}

    for scen in scenario_order:
        #mean +/- std of iteration means
        col_means = pivot_mean[scen].dropna()
        if len(col_means) > 0:
            muA = col_means.mean()
            sdA = col_means.std(ddof=1) if len(col_means) > 1 else 0.0
            summary_A[scen] = f"{muA:.{decimals+1}f} ± {sdA:.{decimals+1}f}"
        else:
            summary_A[scen] = "—"

        #pooled across all runs
        raw_sub = df_raw[(df_raw["scenario"] == scen) &
                         (df_raw["comp"] == comp) &
                         (df_raw["stat"] == stat)]
        vals = raw_sub["value"].dropna()
        if len(vals) > 0:
            muB = vals.mean()
            sdB = vals.std(ddof=1)
            summary_B[scen] = f"{muB:.{decimals+1}f} ± {sdB:.{decimals+1}f}"
        else:
            summary_B[scen] = "—"

        #law of total variance
        raw_sub_iter = raw_sub.groupby("iteration")["value"]
        means = raw_sub_iter.mean()
        vars_ = raw_sub_iter.var(ddof=1)
        if len(means) > 0:
            muC = means.mean()
            varC = vars_.mean() + means.var(ddof=1)
            sdC = np.sqrt(varC)
            summary_C[scen] = f"{muC:.{decimals+1}f} ± {sdC:.{decimals+1}f}"
        else:
            summary_C[scen] = "—"

    formatted.loc["Avg Mean"] = summary_A
    formatted.loc["Avg STD"] = summary_B
    formatted.loc["Avg M+STD"] = summary_C
    formatted.index.name = "Iteration"
    return formatted


table_obs_ks = make_table_with_summary(summary_iter, df, "Observed", "KS", scenario_order, numtimes)
table_exp_ks = make_table_with_summary(summary_iter, df, "Expected", "KS", scenario_order, numtimes)
table_obs_p  = make_table_with_summary(summary_iter, df, "Observed", "p",  scenario_order, numtimes)
table_exp_p  = make_table_with_summary(summary_iter, df, "Expected", "p",  scenario_order, numtimes)



pd.set_option("display.width", 160)
pd.set_option("display.max_columns", None)

print("\nObserved KS")
print(table_obs_ks.to_string())
print()

print("\nExpected KS")
print(table_exp_ks.to_string())
print()
    
print("\nObserved p")
print(table_obs_p.to_string())
print()
    
print("\nExpected p")
print(table_exp_p.to_string())
print()

#table_obs_ks.to_csv("table_obs_ks.csv")
#table_exp_ks.to_csv("table_exp_ks.csv")
#table_obs_p.to_csv("table_obs_p.csv")
#table_exp_p.to_csv("table_exp_p.csv")



Observed KS
                       F-∞           F-1000             F-30             F-TL              L-∞           L-1000             L-30             L-TL
Iteration                                                                                                                                        
1            0.802 ± 0.023    0.520 ± 0.043    0.130 ± 0.020    0.156 ± 0.035    0.834 ± 0.018    0.665 ± 0.036    0.242 ± 0.047    0.175 ± 0.042
2            0.800 ± 0.025    0.518 ± 0.042    0.133 ± 0.024    0.152 ± 0.028    0.830 ± 0.022    0.668 ± 0.033    0.247 ± 0.050    0.177 ± 0.044
3            0.802 ± 0.027    0.515 ± 0.034    0.136 ± 0.024    0.161 ± 0.040    0.830 ± 0.021    0.665 ± 0.029    0.240 ± 0.053    0.179 ± 0.044
4            0.803 ± 0.026    0.518 ± 0.037    0.132 ± 0.021    0.148 ± 0.028    0.832 ± 0.020    0.669 ± 0.032    0.253 ± 0.049    0.185 ± 0.048
5            0.794 ± 0.028    0.516 ± 0.040    0.131 ± 0.023    0.154 ± 0.039    0.828 ± 0.022    0.665 ± 0.034